### Query Expansion

Implements Query Expansion using LangChain.

Workflow:
1. Accept a user's question.
2. Ask the LLM to generate multiple paraphrased versions.
3. Parse the response into structured Pydantic objects.
4. Return the list of rewritten questions.

Query expansion is commonly used in RAG systems to improve document retrieval by
searching with multiple variations of the same question.

In [ ]:
# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

import os                                   # Used to read and set environment variables
from typing import List                     # Used for type hints (list of strings)
from pydantic import BaseModel, Field       # Used to define structured output
from langchain_classic.output_parsers import PydanticToolsParser  # Converts LLM output into Pydantic objects
from langchain_core.prompts import ChatPromptTemplate     # Creates chat prompts
from langchain_openai import ChatOpenAI                  # OpenAI chat model

In [ ]:
# ------------------------------------------------------------
# PYDANTIC MODEL — Defines the structure of LLM output
# ------------------------------------------------------------
# This tells the LLM: "Your response must have a field called
# "paraphrased_query" which is a string."
# LangChain will parse the LLM's tool call into this object.
# ------------------------------------------------------------

class ParaphrasedQuery(BaseModel):
    """
    Represents one paraphrased version of the user's question.
    """

    paraphrased_query: str = Field(
        ...,
        description="A unique paraphrasing of the original question."
    )

In [ ]:
# ------------------------------------------------------------
# ENVIRONMENT SETUP
# ------------------------------------------------------------

from dotenv import load_dotenv
from openai import OpenAI
import os

# Load environment variables from .env file (contains OPENAI_API_KEY)
load_dotenv(dotenv_path="../.env")
api_key = os.getenv("OPENAI_API_KEY")  # Read the API key into a variable
# api_key  # Uncomment to print the key (for debugging only)

# ------------------------------------------------------------
# QUERY EXPANDER CLASS
# ------------------------------------------------------------
# This class orchestrates the entire query expansion pipeline:
#   1. Takes a user question
#   2. Sends it to the LLM with a system prompt asking for paraphrases
#   3. The LLM responds with tool calls (structured as ParaphrasedQuery)
#   4. PydanticToolsParser converts those tool calls into Python objects
#   5. Returns a list of paraphrased strings
# ------------------------------------------------------------

class QueryExpander:

    def __init__(self, api_key: str = None):

        # ----------------------------------------------------
        # SYSTEM PROMPT — Instructions given to the LLM
        # ----------------------------------------------------
        # This prompt tells the LLM HOW to expand the query.
        # Key rules:
        #   - Generate multiple paraphrased versions
        #   - Use synonyms where appropriate
        #   - Don't rephrase acronyms or unfamiliar terms
        #   - Return at least 3 versions
        #   - Preserve the original intent
        # ----------------------------------------------------

        self.system_prompt = """
        You are an expert at expanding user questions into multiple variations.

        Perform query expansion.

        If there are multiple common ways of phrasing a user question
        or common synonyms for key words in the question,
        return multiple rewritten versions.

        If there are acronyms or unfamiliar words,
        do not attempt to rephrase them.

        Return at least 3 versions that preserve the original intent.
        """

        # ----------------------------------------------------
        # CHAT PROMPT TEMPLATE — Wraps system + human message
        # ----------------------------------------------------
        # Structure:
        #   [System Message]  → the system_prompt above
        #   [Human Message]   → the user's question (variable: {question})
        # ----------------------------------------------------

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),  # Role: system, Content: instructions
            ("human", "{question}")          # Role: human, Content: the user's question
        ])

        # ----------------------------------------------------
        # LLM — The language model that generates paraphrases
        # ----------------------------------------------------
        #   model="gpt-4o-mini"  → Fast, cheap model
        #   temperature=0         → Deterministic output (same input = same output)
        # ----------------------------------------------------

        self.llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        )

        # ----------------------------------------------------
        # BIND TOOLS — Attach the Pydantic model as a tool
        # ----------------------------------------------------
        # This tells the LLM: "You can call a function called
        # ParaphrasedQuery with a 'paraphrased_query' string parameter."
        # The LLM will respond with a tool call instead of plain text.
        # ----------------------------------------------------

        self.llm_with_tools = self.llm.bind_tools(
            [ParaphrasedQuery]  # List of Pydantic models the LLM can use
        )

        # ----------------------------------------------------
        # LCEL PIPELINE — Chains prompt → LLM → parser
        # ----------------------------------------------------
        # This is a LangChain Expression Language (LCEL) chain:
        #   Step 1: self.prompt        → Formats the prompt with the user's question
        #   Step 2: self.llm_with_tools → Sends to LLM, gets tool call response
        #   Step 3: PydanticToolsParser → Parses tool calls into ParaphrasedQuery objects
        # ----------------------------------------------------

        self.query_analyzer = (
            self.prompt                    # 1. Format prompt
            | self.llm_with_tools          # 2. Send to LLM
            | PydanticToolsParser(         # 3. Parse output
                tools=[ParaphrasedQuery]   #    into these Pydantic objects
            )
        )

    def expand_query(self, question: str) -> List[str]:
        """
        Takes a user question and returns a list of paraphrased versions.

        Args:
            question: The original question from the user (e.g. "What is Agentic AI?")

        Returns:
            List of paraphrased question strings (e.g. ["Explain Agentic AI.",
            "What does Agentic AI mean?", "Define Agentic AI."])
        """

        try:
            # -------------------------------------------------
            # INVOKE THE PIPELINE
            # -------------------------------------------------
            # This runs the entire LCEL chain:
            #   prompt → LLM → PydanticToolsParser
            # The result is a list of ParaphrasedQuery objects.
            # -------------------------------------------------

            results = self.query_analyzer.invoke({
                "question": question  # Pass the user's question into the {question} variable
            })

            # -------------------------------------------------
            # EXTRACT STRINGS FROM PYDANTIC OBJECTS
            # -------------------------------------------------
            # Each result is a ParaphrasedQuery object.
            # We extract the .paraphrased_query field from each one.
            # -------------------------------------------------

            variations = [
                result.paraphrased_query   # Get the string from each Pydantic object
                for result in results      # Loop over all returned paraphrases
            ]

            return variations  # Return the list of paraphrased strings

        except Exception as e:
            # -------------------------------------------------
            # ERROR HANDLING
            # -------------------------------------------------
            # If anything goes wrong (API error, parsing error, etc.),
            # print the error and return an empty list.
            # -------------------------------------------------

            print(f"Error expanding query: {str(e)}")
            return []  # Return empty list on failure

In [ ]:
# ------------------------------------------------------------
# INSTANTIATE THE QUERY EXPANDER
# ------------------------------------------------------------
# Creates a QueryExpander object.
# This triggers __init__(), which:
#   1. Sets up the system prompt
#   2. Creates the ChatPromptTemplate
#   3. Initializes the LLM (gpt-4o-mini)
#   4. Binds the ParaphrasedQuery tool to the LLM
#   5. Builds the LCEL pipeline (prompt | llm | parser)
# ------------------------------------------------------------

expander = QueryExpander()

In [ ]:
# ------------------------------------------------------------
# DEFINE THE QUESTION
# ------------------------------------------------------------
# The user's original question that we want to expand.
# This will be passed to expander.expand_query().
# ------------------------------------------------------------

# question = "How do I implement RAG using LangChain?"  # Alternative question (commented out)

question = "What is Agentic AI?"  # The question we'll expand

In [ ]:
# ------------------------------------------------------------
# EXPAND THE QUERY
# ------------------------------------------------------------
# This calls the LCEL pipeline:
#   1. Formats the prompt with the question
#   2. Sends to LLM → LLM returns tool calls (ParaphrasedQuery)
#   3. PydanticToolsParser converts tool calls to Python objects
#   4. expand_query() extracts the strings into a list
# ------------------------------------------------------------

variations = expander.expand_query(question)

In [ ]:
# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------
# Print the original question followed by all expanded versions.
# Each variation is a different way to ask the same thing.
# ------------------------------------------------------------

print("Original Question:")
print(question)

print("\nExpanded Queries:")

for i, variation in enumerate(variations, start=1):
    print(f"{i}. {variation}")